<div style="text-align: justify; max-width: 90ch">

# Lecture 03c: Judging the judge

**Status: Mandatory reading.**

In `lec_03b` an LLM judge graded the answers of two variants. When a judge says `yes` to almost every answer, either the system is that good or the judge is easy to please,
and the scores alone cannot tell which. A judge is a measuring instrument, and an instrument is checked against a reference before anyone trusts its readings.

**What this notebook covers**

- The judge as a classifier, checked against reference labels.
- A confusion matrix and the false "yes" rate of a lenient judge.
- What the judge can see: its context window.
- A known bias, and what to do when the judge is not good enough.

</div>

In [1]:
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
from dotenv import load_dotenv
from mlflow.genai.judges import make_judge
from sklearn.metrics import confusion_matrix

from course_helpers.corpus import load_eval_set
from course_helpers.evaluation import (
    FACTS_RUBRIC,
    JUDGE_INPUT_TOKENS,
    build_eval_records_from,
    keyword_coverage,
    make_facts_judge,
    model_with_window,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** No RAG chain this time: the answers to grade are fixed text, stored in a file (section 2). From the `course_helpers` helpers:

- the 19 questions as MLflow records, looked up by question id (`q01` ... `q19`);
- the judge of `lec_03b`, `make_facts_judge`: the rubric `FACTS_RUBRIC`, temperature 0, thinking off, on the chat model.

</div>

In [3]:
eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")
records_by_id = {
    row["id"]: record
    for row, record in zip(eval_set, build_eval_records_from(eval_set))
}
judge = make_facts_judge(CHAT_MODEL, OLLAMA_HOST)

print(f"{len(records_by_id)} records; judge model: {CHAT_MODEL}")

19 records; judge model: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. The judge is a classifier

A yes/no judge predicts `yes` or `no`, so it is measured like any classifier, against true labels; here they are written against the rubric:

| Term | Meaning |
| --- | --- |
| **reference label** | the verdict an answer should get by the rubric, written down before any judge sees the answer |
| **agreement** | share of answers where judge and label agree: the accuracy of a classifier |
| **false yes** | the judge passes an answer labelled `no`: the judge is **lenient** |
| **false no** | the judge fails an answer labelled `yes`: the judge is **strict** |
| **false-yes rate** | share of the answers labelled `no` that the judge passes |
| **bias** | a systematic pull on the verdict from something the rubric does not ask about |

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Answers with reference labels

A lenient judge is the dangerous one: it reports a good system whatever the system does. `corpus/eval/judge_labels.jsonl` holds 26 answers to questions of the evaluation set,
each with a reference label written against the rubric of `lec_03b`: `yes` only if the answer states every expected fact and contradicts none.
Some answers were written to be **right**, **paraphrased**, **reversed**, **partial** or wrong in one **detail**; the **real answers** come from the `lec_02d` system in one example run.
The answers are stored, not generated, so every label holds on every run.

**Draft:** the answers and labels were written for this notebook and await the instructor's review (the file's `status` column);
question them as you read.

This notebook uses eleven of them, fixed by id; the `why` column says why an answer fails:

</div>

In [4]:
labelled = pd.read_json(REPO / "corpus" / "eval" / "judge_labels.jsonl", lines=True)
SUBSET = ["j02", "j04", "j06", "j13", "j14", "j15", "j16", "j18", "j21", "j22", "j26"]
answers = labelled[labelled["id"].isin(SUBSET)].reset_index(drop=True)

answers[["id", "kind", "label", "why"]]

,id,kind,label,why
0,j02,paraphrase,yes,states every expected fact
1,j04,partial,no,says nothing about lists
2,j06,reversed,no,penalties swapped
3,j13,right,yes,states every expected fact
4,j14,reversed,no,weight grows with distance
5,j15,right,yes,states every expected fact
6,j16,wrong detail,no,wrong platform
7,j18,wrong detail,no,normality replaced by multicollinearity
8,j21,real answer,no,does not mention n_iter
9,j22,real answer,no,"no Euclidean distance, no StandardScaler"


<div style="text-align: justify; max-width: 90ch">

Seven of the eleven are labelled `no`, on purpose: a set of right answers only would never catch a judge that passes everything.
One real answer, with the facts it is checked against:

</div>

In [5]:
real = answers.set_index("id").loc["j21"]
record = records_by_id[real["question_id"]]

print(f"Question ({real['question_id']}): {record['inputs']['question']}")
print(f"Expected facts: {record['expectations']['expected_facts']}")
print(f"\nAnswer: {real['answer']}")
print(f"\nLabel: {real['label']} | {real['why']}")

Question (q03): When should I use RandomizedSearchCV instead of GridSearchCV?
Expected facts: ['When the grid is large or a hyperparameter is continuous.', 'It samples a fixed number of combinations, set by n_iter.']

Answer: When the grid is **large** (i.e., the number of parameter combinations is too big to try them all), you should use `RandomizedSearchCV` instead of `GridSearchCV`. This is because `RandomizedSearchCV` samples a **fixed number** of combinations from the space, which is more efficient for large grids. It finds a near-best setting for a fraction of the fits, making it suitable when you cannot afford to try every possible combination.

Label: no | does not mention n_iter


<div style="text-align: justify; max-width: 90ch">

The answer says "a fixed number" of combinations but never names `n_iter`, so by the rubric it is `no`. A reader might call it good enough;
a label makes that strictness explicit. When two people label the same answer differently, the rubric is unclear, and no judge can be better than the rubric.

### ⏸ Pause point

Would you label this answer `no`? If not, how would you rewrite the expected facts of this question (q03) so that you and a colleague always agree?

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Two judges that never change their mind

Two reference points first, whose verdicts are fixed by code, so every number in this section is the same on every run:

- **always yes**: a judge that has learnt nothing;
- **keywords**: the keyword scorer of `lec_03a` (`keyword_coverage`), read as `yes` when the keywords of every fact are found.

One function compares any list of verdicts with the labels:

</div>

In [6]:
def measure_agreement_between(labels, verdicts):
    """Agreement, false-yes rate and false-no rate of verdicts against labels."""
    labels = pd.Series(list(labels))
    verdicts = pd.Series(list(verdicts))
    return pd.Series(
        {
            "agreement": (labels == verdicts).mean(),
            "false_yes_rate": (verdicts[labels == "no"] == "yes").mean(),
            "false_no_rate": (verdicts[labels == "yes"] != "yes").mean(),
        }
    )

In [7]:
always_yes = ["yes"] * len(answers)

keywords = []
for question_id, answer in zip(answers["question_id"], answers["answer"]):
    facts = records_by_id[question_id]["expectations"]["expected_facts"]
    keywords.append("yes" if keyword_coverage(answer, facts) == 1 else "no")

reports = pd.DataFrame(
    {
        "always yes": measure_agreement_between(answers["label"], always_yes),
        "keywords": measure_agreement_between(answers["label"], keywords),
    }
)

reports.round(2)

,always yes,keywords
agreement,0.36,0.45
false_yes_rate,1.00,0.71
false_no_rate,0.00,0.25


<div style="text-align: justify; max-width: 90ch">

Read the `always yes` column first. Its agreement is the share of `yes` labels: on a set of mostly right answers it would look excellent,
while its false-yes rate of 1 says it passes every wrong answer. **Agreement alone flatters a lenient judge**; read the false-yes rate next to it.
The keyword column is the scorer of `lec_03a` in numbers: it passes the reversed answers and fails the paraphrase. Any judge worth its cost has to beat both columns.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. The LLM judge against the labels

`run_judge_on` asks the judge once per answer, with the question and expected facts of its record, and returns the verdicts in row order:
eleven model calls. The cell prints how long they took; expect several times longer without a GPU.

</div>

In [8]:
def run_judge_on(judge, labelled, records_by_id):
    """One verdict of `judge` per labelled answer, as text, in row order."""
    verdicts = []

    for question_id, answer in zip(labelled["question_id"], labelled["answer"]):
        record = records_by_id[question_id]
        feedback = judge(
            inputs=record["inputs"],
            outputs=answer,
            expectations=record["expectations"],
        )
        verdicts.append(str(feedback.value))

    return verdicts


started = time.time()
verdicts = run_judge_on(judge, answers, records_by_id)

print(f"{len(verdicts)} verdicts in {time.time() - started:.0f} s")

11 verdicts in 7 s


<div style="text-align: justify; max-width: 90ch">

A **confusion matrix** ([scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html),
from the classification lectures) counts each pair of label and verdict; `labels=["yes", "no"]` fixes the order. Rows are the labels, columns the verdicts:
the lower-left cell counts false yes, the upper-right false no.

</div>

In [9]:
pd.DataFrame(
    confusion_matrix(answers["label"], verdicts, labels=["yes", "no"]),
    index=["label yes", "label no"],
    columns=["judge yes", "judge no"],
)

,judge yes,judge no
label yes,4,0
label no,4,3


<div style="text-align: justify; max-width: 90ch">

Read the lower-left cell. In example runs with `qwen3:1.7b` it held the errors: the judge passed most of the wrong answers and failed none of the right ones.
That is leniency: if a judge column in `lec_03b` was mostly `yes`, it may say more about the judge than the answers. The three rates, next to the two reference judges:

</div>

In [10]:
reports[CHAT_MODEL] = measure_agreement_between(answers["label"], verdicts)

reports.round(2)

,always yes,keywords,qwen3:1.7b
agreement,0.36,0.45,0.64
false_yes_rate,1.00,0.71,0.57
false_no_rate,0.00,0.25,0.00


<div style="text-align: justify; max-width: 90ch">

Compare the judge's column with the reference judges. The rates do not say *which* answers it got wrong; one row per answer does:

</div>

In [11]:
graded = answers.assign(verdict=verdicts)

graded[["id", "kind", "label", "verdict"]]

,id,kind,label,verdict
0,j02,paraphrase,yes,yes
1,j04,partial,no,no
2,j06,reversed,no,yes
3,j13,right,yes,yes
4,j14,reversed,no,yes
5,j15,right,yes,yes
6,j16,wrong detail,no,no
7,j18,wrong detail,no,no
8,j21,real answer,no,yes
9,j22,real answer,no,yes


<div style="text-align: justify; max-width: 90ch">

Look for rows with label `no` and verdict `yes`: what kind of answer are they? In example runs with `qwen3:1.7b`, the judge passed the two reversed answers and the two real answers that miss a fact,
and caught the other three wrong answers: better than the keywords, far from a careful reader. A missing fact in a real answer is the mistake the RAG system actually makes.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Ask once more

`lec_03b` asked its judge once more on one answer. Here, once more on the real answer of section 2: compare the two verdicts.

</div>

In [12]:
j21 = answers[answers["id"] == "j21"]
again = run_judge_on(judge, j21, records_by_id)

print(f"first: {verdicts[j21.index[0]]} | again: {again[0]}")

first: yes | again: yes


<div style="text-align: justify; max-width: 90ch">

At temperature 0 the judge takes its most likely token, so the two verdicts usually match; in example runs with `qwen3:1.7b` they matched, and both were wrong.
Not always, though: in example runs, a reversed answer to q19 got `no` once and `yes` another time, because small numeric differences can tip a close call.
**Repeatable is not the same as right**: asking again detects noise, never bias. Only the labels detect bias.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. What the judge can see

The judge reaches Ollama through the OpenAI-compatible endpoint, which takes no `num_ctx`. It therefore works in the server's default window:
4,096 tokens unless the GPU has 24 GB of VRAM or more (`lec_01a`). A longer prompt is cut silently, from the start, and the start is where the rubric sits.

MLflow records how many tokens each verdict read, in the metadata of the feedback. The longest answer of section 2, graded once more:

</div>

In [13]:
longest = answers.loc[answers["answer"].str.len().idxmax()]
longest_record = records_by_id[longest["question_id"]]
feedback = judge(
    inputs=longest_record["inputs"],
    outputs=longest["answer"],
    expectations=longest_record["expectations"],
)
tokens_read = feedback.metadata[JUDGE_INPUT_TOKENS]

print(
    f"{longest['id']}: {len(longest['answer'])} characters, {tokens_read} tokens read by the judge"
)

j22: 798 characters, 467 tokens read by the judge


<div style="text-align: justify; max-width: 90ch">

A question, an answer of a few hundred tokens and the expected facts fit with room to spare. A judge that reads a whole trace, the retrieved chunks or a long conversation may not, and nothing will warn you.

⏱ **Skip if running long.**

A judge that reads a whole lecture file, as you might to check an answer against its source: lecture 13c, the source of q04, given to the judge as the "answer".
First the obvious attempt, a larger window in `inference_params`:

</div>

In [14]:
lecture_dir = REPO / "corpus" / "uoa_py_course" / "lecture_13"
lecture_text = (lecture_dir / "lec_13c_mlflow_tracking_basics.md").read_text()
q04_record = records_by_id["q04"]

# WRONG: the OpenAI-compatible endpoint ignores num_ctx
judge_default_window = make_judge(
    name="states_facts",
    instructions=FACTS_RUBRIC,
    model=f"ollama:/{CHAT_MODEL}",
    feedback_value_type=Literal["yes", "no"],
    inference_params={"temperature": 0, "reasoning_effort": "none", "num_ctx": 8192},
    base_url=f"{OLLAMA_HOST}/v1/chat/completions",
)
feedback = judge_default_window(
    inputs=q04_record["inputs"],
    outputs=lecture_text,
    expectations=q04_record["expectations"],
)

tokens_read = feedback.metadata["mlflow.assessment.judgeInputTokens"]
print(f"verdict: {feedback.value} | tokens read: {tokens_read}")

verdict: yes | tokens read: 2050


<div style="text-align: justify; max-width: 90ch">

A verdict and no error. Compare the count with the next cell's: far below means Ollama cut the prompt, and the rubric was in the part it dropped, so the judge graded without it.
(On a GPU with 24 GB of VRAM or more the default window may already be large enough.)

The fix is on the server side: a copy of the model with a larger window. `model_with_window` calls `client.create` ([Ollama API](https://docs.ollama.com/api/create)),
which makes the copy from the original with one changed setting: same weights, nothing downloaded, and it shows up in `ollama list`.
Running it again only updates the copy. `make_facts_judge` builds the judge of `lec_03b` on any model.

</div>

In [15]:
client = ollama.Client(host=OLLAMA_HOST)
JUDGE_MODEL_8K = model_with_window(client, CHAT_MODEL, 8192)

# RIGHT: the window is a property of the model the judge calls
judge_8k = make_facts_judge(JUDGE_MODEL_8K, OLLAMA_HOST)
feedback = judge_8k(
    inputs=q04_record["inputs"],
    outputs=lecture_text,
    expectations=q04_record["expectations"],
)

tokens_read = feedback.metadata["mlflow.assessment.judgeInputTokens"]
print(f"{JUDGE_MODEL_8K} read {tokens_read} tokens")

qwen3:1.7b-ctx8k read 6057 tokens


<div style="text-align: justify; max-width: 90ch">

Compare the two token counts: the copy read the whole prompt. The price is memory: the copy is a second model for Ollama,
which keeps it loaded next to the original if memory allows, with room for 8,192 tokens (`lec_01a`). A judge needs the window of its longest prompt;
count its tokens, as above, before trusting a verdict.

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. A known bias: a source line

⏱ **Skip if running long.**

Studies of LLM judges ([Zheng et al., 2023](https://arxiv.org/abs/2306.05685)) found that judges reward text unrelated to correctness,
such as length. The RAG prompt asks every answer to name its source file, and a source line is such text. Test it on the written wrong answers the judge caught above, each with the line `Source: <expected file>` added.
The facts stay wrong, so the label stays `no`. One model call per answer; the table shows each verdict before and after:

</div>

In [16]:
caught = graded[
    (graded["kind"] != "real answer")
    & (graded["label"] == "no")
    & (graded["verdict"] == "no")
]
sources = [
    records_by_id[q]["expectations"]["expected_source"] for q in caught["question_id"]
]
with_source = caught.assign(
    answer=[f"{a}\n\nSource: {s}" for a, s in zip(caught["answer"], sources)]
)

caught[["id", "kind", "verdict"]].assign(
    with_source_line=run_judge_on(judge, with_source, records_by_id)
)

,id,kind,verdict,with_source_line
1,j04,partial,no,yes
6,j16,wrong detail,no,no
7,j18,wrong detail,no,yes


<div style="text-align: justify; max-width: 90ch">

Look for a `no` that turns into `yes`: a line that says nothing about the facts changed the verdict. In example runs with `qwen3:1.7b`, the judge caught three written wrong answers and two of them flipped;
on the 11 written wrong answers of the file, the false-yes rate rose from about 0.65 to 0.91, while `qwen3:8b` held every `no`.
A judge that rewards a source line rewards the RAG system for a habit, not for a fact.

</div>

<div style="text-align: justify; max-width: 90ch">

## 8. Is the judge good enough?

A judge is good enough to compare two variants when it is not lenient on the mistakes your system makes, its error is smaller than the difference you want to see,
and extra text does not move it. In example runs `qwen3:1.7b` failed the first and the third; for the second, compare its false-yes rate with the gap between the variants in `lec_03b`.

</div>

<div style="text-align: justify; max-width: 90ch">

When a judge fails, there are four ways out, each with a price:

| Option | What it costs |
| --- | --- |
| **a bigger judge**: in example runs on all 26 answers, `qwen3:8b` agreed with every label | memory and seconds per verdict; a hosted judge costs money per token |
| **one fact per call**: a stricter rubric, one expected fact per verdict | more calls; measure the change against the same labels |
| **thinking on** (`lec_01b`) | more tokens and seconds per verdict; measure it against the same labels too |
| **human spot checks**: read a sample of `yes` verdicts, label where two judges differ | a person's time, the most expensive resource |

In example runs, thinking on lowered the false-yes rate of `qwen3:1.7b`, yet it still passed the real answers with a missing fact.
MLflow can also tune a judge's instructions from labelled feedback ([judge alignment](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/alignment/)).

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

You must choose between the two variants of `lec_03b` by tomorrow. Which judge do you use, which answers do you read yourself,
and what do you write in the decision record about the judge?

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A judge is a classifier: check it against **reference labels** written against the rubric, with agreement, a **confusion matrix** and the **false-yes rate**.
- **Agreement alone flatters a lenient judge**; `always yes` and the keyword scorer are reference points whose numbers never change.
- In example runs `qwen3:1.7b` was **lenient**, also on real answers with a missing fact; asking again catches noise, not bias.
- The judge sees only its **window**, and loses the start of a longer prompt: count its tokens; a model copy has a larger window.
- A **source line** alone could turn a small judge's `no` into `yes`; in example runs a bigger judge held.
- When the judge is not good enough: a bigger judge, one fact per call, thinking on, or human spot checks.

The helpers `run_judge_on`, `measure_agreement_between`, `judge_input_tokens` and `model_with_window` are in `course_helpers.evaluation`.

**Next: `lec_03d`**, where the model stops only answering and starts acting: tools and agents.

</div>